# 04. Nonlinear Modelling

## 1. Notebook Objective and Modelling Framework

Notebook 3 established the probabilistic baseline for the Premier League Probability Engine. The selected multinomial logistic-regression model demonstrated that the engineered pre-match features contain predictive information beyond naive outcome-frequency forecasts.

The purpose of this notebook is to test whether nonlinear models can improve those probability estimates by learning interactions and threshold effects that a linear decision boundary cannot represent directly.

The central research question is:

> Can nonlinear models exploit interactions within the engineered pre-match features while improving out-of-sample probability forecasts relative to the tuned logistic-regression baseline?

The prediction target remains:

$$
Y_i \in \{H,D,A\},
$$

where:

- $H$ represents a home win;
- $D$ represents a draw;
- $A$ represents an away win.

Every model must return a complete probability vector in the fixed project order:

$$
\widehat{\mathbf{p}}_i
=
\left(
\widehat{p}_{i,H},
\widehat{p}_{i,D},
\widehat{p}_{i,A}
\right),
$$

subject to:

$$
\widehat{p}_{i,H}
+
\widehat{p}_{i,D}
+
\widehat{p}_{i,A}
=
1.
$$

### Evaluation discipline

The chronological split from Notebook 3 will be reproduced exactly:

- training set: all seasons before the final two completed seasons;
- validation set: the second-most-recent completed season;
- test set: the most-recent completed season.

Model families and hyperparameters must be selected using validation performance only. The principal selection metric remains multiclass log loss, with Brier score and accuracy used as supporting diagnostics.

The tuned logistic-regression validation benchmark from Notebook 3 is:

| Metric | Validation result |
|---|---:|
| Log loss | 0.934551 |
| Brier score | 0.549322 |
| Accuracy | 0.592105 |

The nonlinear models must therefore improve probability quality rather than merely produce different predicted classes.

### Planned model sequence

This notebook will examine:

1. a Random Forest probability baseline;
2. a gradient-boosted tree model;
3. validation-based hyperparameter tuning;
4. feature-importance and error diagnostics;
5. final comparison with the tuned logistic-regression benchmark.

> **Run instruction:** select the project virtual environment as the kernel, then use **Restart Kernel and Run All Cells**. The transferred setup cells are designed to reproduce Notebook 3's data, split and benchmark framework exactly.

## 2. Load and Validate the Processed Modelling Dataset

The modelling dataset created in `02_feature_engineering.ipynb` will now be loaded from the project’s processed-data directory.

The preferred input is:

`data/processed/premier_league_model_data.parquet`

Parquet is used because it preserves numeric, nullable-integer and date-related data types more reliably than CSV. The CSV export will remain available as a fallback if the Parquet file cannot be loaded.

Before modelling begins, the dataset must be checked to confirm that:

- the file exists in the expected project directory;
- the dataset contains at least one fixture;
- column names are unique;
- exact duplicate rows are absent;
- the identifier columns are available;
- the target column is present;
- the target contains only `H`, `D` and `A`;
- fixture dates can be parsed successfully;
- rows remain chronologically ordered within each season;
- the exported dataset contains no infinite numeric values.

The notebook will locate the repository root dynamically rather than relying on the notebook’s current working directory. This prevents paths such as `notebooks/data/processed/` from being created accidentally when the notebook is executed from inside the `notebooks` directory.

At this stage, no rows will be removed and no predictors will be transformed. The objective is only to verify that the exported feature-engineering output can be treated as the fixed input for the modelling pipeline.

A successful validation will establish the following primary objects:

- `model_data`: the complete processed modelling dataset;
- `season_column`: the season identifier;
- `date_column`: the fixture date;
- `home_team_column`: the home-team identifier;
- `away_team_column`: the away-team identifier;
- `target_column`: the full-time match result.

These objects will be used throughout the remainder of the notebook.

In [ ]:
# ============================================================
# 2. Load and Validate the Processed Modelling Dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate the project root
# ------------------------------------------------------------

def find_project_root(start_path):
    """
    Find the nearest parent directory containing the Git repository.
    """
    start_path = Path(start_path).resolve()

    for directory in [start_path, *start_path.parents]:
        if (directory / ".git").exists():
            return directory

    raise FileNotFoundError(
        "Could not locate the project root containing the .git directory."
    )


project_root = find_project_root(Path.cwd())

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

parquet_path = (
    processed_data_directory
    / "premier_league_model_data.parquet"
)

csv_path = (
    processed_data_directory
    / "premier_league_model_data.csv"
)


# ------------------------------------------------------------
# Load the modelling dataset
# ------------------------------------------------------------

if parquet_path.exists():
    model_data = pd.read_parquet(parquet_path)
    loaded_file = parquet_path
    loaded_format = "Parquet"

elif csv_path.exists():
    model_data = pd.read_csv(csv_path)
    loaded_file = csv_path
    loaded_format = "CSV"

else:
    raise FileNotFoundError(
        "Could not find the processed modelling dataset.\n\n"
        f"Checked:\n- {parquet_path}\n- {csv_path}"
    )


# ------------------------------------------------------------
# Identify essential columns
# ------------------------------------------------------------

def find_first_existing_column(
    dataframe,
    candidates,
    label,
    required=True,
):
    """
    Return the first candidate column present in the DataFrame.
    """
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not identify the {label} column. "
            f"Checked: {candidates}"
        )

    return None


season_column = find_first_existing_column(
    model_data,
    ["Season", "season"],
    label="season",
)

date_column = find_first_existing_column(
    model_data,
    ["Date", "date", "MatchDate", "match_date"],
    label="fixture date",
)

home_team_column = find_first_existing_column(
    model_data,
    ["HomeTeam", "home_team", "Home"],
    label="home-team",
)

away_team_column = find_first_existing_column(
    model_data,
    ["AwayTeam", "away_team", "Away"],
    label="away-team",
)

target_column = find_first_existing_column(
    model_data,
    ["FTR", "Result", "result", "FullTimeResult"],
    label="target",
)


# ------------------------------------------------------------
# Basic structural validation
# ------------------------------------------------------------

assert len(model_data) > 0, (
    "The modelling dataset is empty."
)

assert model_data.columns.is_unique, (
    "The modelling dataset contains duplicate column names."
)

exact_duplicate_rows = int(
    model_data.duplicated().sum()
)

assert exact_duplicate_rows == 0, (
    f"{exact_duplicate_rows} exact duplicate rows were detected."
)


# ------------------------------------------------------------
# Parse and validate fixture dates
# ------------------------------------------------------------

model_data[date_column] = pd.to_datetime(
    model_data[date_column],
    errors="coerce",
    dayfirst=True,
)

assert model_data[date_column].notna().all(), (
    "At least one fixture date could not be parsed."
)


# ------------------------------------------------------------
# Validate fixture identifiers
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

assert model_data[identifier_columns].notna().all().all(), (
    "One or more fixture identifier columns contain missing values."
)

assert (
    model_data[home_team_column]
    != model_data[away_team_column]
).all(), "A fixture cannot contain the same home and away team."

duplicate_fixture_count = int(
    model_data.duplicated(
        subset=identifier_columns,
    ).sum()
)

assert duplicate_fixture_count == 0, (
    f"{duplicate_fixture_count} duplicate fixtures were detected."
)


# ------------------------------------------------------------
# Validate the target
# ------------------------------------------------------------

model_data[target_column] = (
    model_data[target_column]
    .astype("string")
    .str.strip()
    .str.upper()
)

valid_target_values = {"H", "D", "A"}

invalid_target_values = sorted(
    set(
        model_data[target_column]
        .dropna()
        .unique()
    )
    - valid_target_values
)

assert not invalid_target_values, (
    "The target contains invalid values: "
    f"{invalid_target_values}"
)

assert model_data[target_column].notna().all(), (
    "The target contains missing values."
)


# ------------------------------------------------------------
# Confirm chronological ordering within seasons
# ------------------------------------------------------------

chronology_check = (
    model_data
    .groupby(
        season_column,
        sort=False,
    )[date_column]
    .apply(lambda dates: dates.is_monotonic_increasing)
)

assert chronology_check.all(), (
    "Fixtures are not chronologically ordered within every season."
)


# ------------------------------------------------------------
# Check numeric columns for infinite values
# ------------------------------------------------------------

numeric_columns = model_data.select_dtypes(
    include=[np.number]
).columns.tolist()

infinite_value_counts = {
    column: int(
        np.isinf(
            pd.to_numeric(
                model_data[column],
                errors="coerce",
            ).astype(float)
        ).sum()
    )
    for column in numeric_columns
}

columns_with_infinite_values = {
    column: count
    for column, count in infinite_value_counts.items()
    if count > 0
}

assert not columns_with_infinite_values, (
    "Infinite values were detected: "
    f"{columns_with_infinite_values}"
)


# ------------------------------------------------------------
# Create compact validation summaries
# ------------------------------------------------------------

target_distribution = (
    model_data[target_column]
    .value_counts()
    .reindex(["H", "D", "A"])
    .rename_axis("Outcome")
    .reset_index(name="Fixtures")
)

target_distribution["Percentage"] = (
    100
    * target_distribution["Fixtures"]
    / len(model_data)
).round(2)

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Fixtures",
            "Columns",
            "Seasons",
            "Earliest fixture",
            "Latest fixture",
            "Numeric columns",
            "Columns with missing values",
        ],
        "Value": [
            f"{len(model_data):,}",
            len(model_data.columns),
            model_data[season_column].nunique(),
            model_data[date_column].min().date(),
            model_data[date_column].max().date(),
            len(numeric_columns),
            int(model_data.isna().any().sum()),
        ],
    }
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Processed modelling dataset loaded successfully.")
print(f"Format: {loaded_format}")
print(f"File: {loaded_file}")
print(f"Shape: {model_data.shape}")
print(f"Target column: {target_column}")

display(dataset_summary)
display(target_distribution)
display(model_data.head(10))

### Results and Interpretation

The processed modelling dataset loaded successfully from the project-level `data/processed` directory.

The validation confirms that:

- the dataset contains fixtures and modelling variables;
- all column names are unique;
- no exact duplicate rows are present;
- fixture identifiers are complete;
- no fixture contains the same home and away team;
- the target column contains only `H`, `D` and `A`;
- fixture dates were parsed successfully;
- fixtures remain chronologically ordered within each season;
- no numeric column contains infinite values.

The target distribution provides the first indication of class imbalance in Premier League outcomes.

Home wins are generally the most common result, reflecting the historical home advantage. Draws and away wins occur less frequently, meaning that accuracy alone would be an incomplete measure of model quality.

A model could achieve a superficially reasonable accuracy by predicting the most common outcome too often while still producing poor probabilities for draws and away wins. For this reason, later evaluation will prioritise multiclass log loss and Brier score.

The successfully loaded `model_data` DataFrame now represents the fixed input for the baseline-modelling pipeline.

No rows have been removed and no feature transformations have yet been applied.

## 3. Define Identifiers, Target and Predictor Columns

Before constructing the training, validation and test sets, the columns in `model_data` must be separated according to their modelling role.

The dataset contains three distinct groups:

1. fixture identifiers;
2. the prediction target;
3. eligible pre-match predictors.

### Fixture Identifiers

The identifier columns describe each match and are retained for chronological splitting, interpretation and prediction output:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These columns will not be passed directly into the baseline logistic-regression model.

`Season` and `Date` are required to preserve the temporal structure of the dataset. The team-name columns identify each fixture but are excluded from the initial baseline because representing team identity directly would require an additional categorical-encoding strategy.

Team strength is already represented through quantitative pre-match variables such as Elo ratings, rolling form and league-table state.

### Target Variable

The prediction target is the full-time result:

`FTR`

with possible values:

$$
Y_i \in \{H,D,A\}.
$$

The classes represent:

- `H`: home win;
- `D`: draw;
- `A`: away win.

For consistent probability output, the class order used throughout the project will be:

$$
(H,D,A).
$$

Maintaining a fixed class order is essential because each predicted-probability column must always correspond to the same outcome.

### Predictor Variables

The predictor set consists of the remaining eligible numeric columns in the processed modelling dataset.

These variables describe information available before kickoff, including:

- pre-match Elo ratings;
- general rolling form;
- venue-specific rolling form;
- rest and fixture congestion;
- relative home–away differences;
- season progress;
- reconstructed pre-match league-table state;
- league-position category indicators.

The predictor matrix will be denoted by:

$$
X \in \mathbb{R}^{N \times P},
$$

where:

- $N$ is the number of fixtures;
- $P$ is the number of predictor variables.

The target vector will be denoted by:

$$
\mathbf{y}
=
(y_1,\ldots,y_N).
$$

### Leakage Protection

The predictor set must not contain:

- `FTR`;
- full-time or half-time goals;
- match statistics recorded during the fixture;
- final-season information;
- bookmaker probabilities or odds;
- temporary feature-engineering columns.

The processed dataset was designed to exclude these variables, but this notebook will validate the separation again before modelling.

### Missing Values

Some predictors contain intentional missing values.

For example, league position and league-position category indicators are undefined before the first completed fixture batch of each season.

Later preprocessing will impute missing numeric values using statistics learned from the training set only.

No imputation, scaling or other transformation will be fitted before the chronological split. This prevents information from the validation or test periods from influencing the training pipeline.

This section will create the following objects:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`

It will also produce a feature summary confirming the number, data type and missingness of the available predictors.

In [ ]:
# ============================================================
# 3. Define Identifiers, Target and Predictor Columns
# ============================================================

# ------------------------------------------------------------
# Define identifier and target columns
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

class_order = ["H", "D", "A"]


# ------------------------------------------------------------
# Identify eligible numeric predictor columns
# ------------------------------------------------------------

excluded_columns = set(
    identifier_columns
    + [target_column]
)

feature_columns = [
    column
    for column in model_data.columns
    if (
        column not in excluded_columns
        and pd.api.types.is_numeric_dtype(model_data[column])
    )
]


# ------------------------------------------------------------
# Validate that predictors have been identified
# ------------------------------------------------------------

assert feature_columns, (
    "No numeric predictor columns were identified."
)

assert len(feature_columns) == len(set(feature_columns)), (
    "The predictor list contains duplicate column names."
)

assert target_column not in feature_columns, (
    "The target column has entered the predictor set."
)

assert not set(identifier_columns).intersection(feature_columns), (
    "One or more identifier columns entered the predictor set."
)


# ------------------------------------------------------------
# Check for obvious leakage columns
# ------------------------------------------------------------

blocked_exact_names = {
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
    "HomeGoals",
    "AwayGoals",
    "HomeScore",
    "AwayScore",
    "FullTimeResult",
}

blocked_name_fragments = [
    "bookmaker",
    "market_probability",
    "marketprob",
    "implied_probability",
    "impliedprob",
    "final_position",
    "finalposition",
]

blocked_predictors = [
    column
    for column in feature_columns
    if (
        column in blocked_exact_names
        or any(
            fragment in column.lower()
            for fragment in blocked_name_fragments
        )
    )
]

assert not blocked_predictors, (
    "Potential leakage columns were detected in the predictor set: "
    f"{blocked_predictors}"
)


# ------------------------------------------------------------
# Construct the predictor matrix and target vector
# ------------------------------------------------------------

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

fixture_metadata = model_data[
    identifier_columns
].copy()


# ------------------------------------------------------------
# Validate shapes and index alignment
# ------------------------------------------------------------

assert len(X) == len(model_data), (
    "The predictor matrix does not contain every fixture."
)

assert len(y) == len(model_data), (
    "The target vector does not contain every fixture."
)

assert X.index.equals(y.index), (
    "The predictor matrix and target vector are not aligned."
)

assert fixture_metadata.index.equals(X.index), (
    "Fixture metadata and predictors are not aligned."
)

assert X.columns.is_unique, (
    "The predictor matrix contains duplicate columns."
)


# ------------------------------------------------------------
# Validate target classes
# ------------------------------------------------------------

observed_classes = set(
    y.dropna().unique()
)

assert observed_classes == set(class_order), (
    "The observed target classes do not match H, D and A. "
    f"Observed classes: {sorted(observed_classes)}"
)


# ------------------------------------------------------------
# Validate predictor values
# ------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in feature_columns
    if not pd.api.types.is_numeric_dtype(X[column])
]

assert not non_numeric_predictors, (
    "Non-numeric predictor columns were detected: "
    f"{non_numeric_predictors}"
)

infinite_predictor_counts = {}

for column in feature_columns:
    numeric_values = pd.to_numeric(
        X[column],
        errors="coerce",
    ).astype(float)

    infinite_predictor_counts[column] = int(
        np.isinf(numeric_values).sum()
    )

predictors_with_infinite_values = {
    column: count
    for column, count in infinite_predictor_counts.items()
    if count > 0
}

assert not predictors_with_infinite_values, (
    "Infinite values were detected in the predictors: "
    f"{predictors_with_infinite_values}"
)


# ------------------------------------------------------------
# Create the feature summary
# ------------------------------------------------------------

feature_summary = pd.DataFrame(
    {
        "Feature": feature_columns,
        "DataType": [
            str(X[column].dtype)
            for column in feature_columns
        ],
        "MissingValues": [
            int(X[column].isna().sum())
            for column in feature_columns
        ],
        "MissingPercentage": [
            round(
                100 * X[column].isna().mean(),
                2,
            )
            for column in feature_columns
        ],
        "UniqueValues": [
            int(X[column].nunique(dropna=True))
            for column in feature_columns
        ],
        "Minimum": [
            X[column].min(skipna=True)
            for column in feature_columns
        ],
        "Maximum": [
            X[column].max(skipna=True)
            for column in feature_columns
        ],
    }
)

features_with_missing_values = (
    feature_summary[
        feature_summary["MissingValues"] > 0
    ]
    .sort_values(
        by=[
            "MissingPercentage",
            "MissingValues",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Identifiers, target and predictors defined successfully.")
print(f"Fixtures: {len(model_data):,}")
print(f"Identifier columns: {len(identifier_columns)}")
print(f"Target column: {target_column}")
print(f"Predictor columns: {len(feature_columns)}")
print(
    "Predictors containing missing values:",
    len(features_with_missing_values),
)
print(f"Class order: {class_order}")

display(feature_summary)

if not features_with_missing_values.empty:
    display(features_with_missing_values)

### Results and Interpretation

The modelling columns have now been separated into fixture identifiers, the prediction target and eligible numeric predictors.

The identifier columns are:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These variables are retained for chronological splitting, fixture tracking and interpretation, but they are not passed directly into the baseline logistic-regression model.

The target variable is:

- `FTR`

with the fixed class order:

$$
(H,D,A).
$$

This ordering will be preserved whenever predicted probabilities are stored or evaluated, ensuring that each probability column always corresponds to the correct match outcome.

The predictor matrix `X` contains the numeric pre-match variables created during feature engineering, while the target vector `y` contains the observed full-time results.

The validation confirms that:

- the target is not included among the predictors;
- no fixture identifier is included among the predictors;
- every predictor is numeric;
- no predictor contains infinite values;
- predictor names are unique;
- `X`, `y` and `fixture_metadata` contain the same fixtures in the same order;
- all three target classes are present;
- no obvious post-match, bookmaker or final-season variables entered the predictor set.

Some predictor columns contain missing values. These are expected for features that require previous match information or a meaningful pre-match league table.

Missing values have not yet been imputed. Imputation must be learned from the training set only after the chronological split, preventing information from the validation or test periods from influencing the preprocessing pipeline.

The objects now available for modelling are:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`
- `fixture_metadata`

The next stage is to divide the fixtures into chronological training, validation and test periods.

## 4. Chronological Train–Validation–Test Split

The modelling dataset must now be divided into separate training, validation and test periods.

Because football fixtures occur through time, the split must preserve chronology. A random split would allow the model to train on matches played after fixtures contained in the validation or test sets, producing an unrealistic estimate of future performance.

The split will therefore be performed using complete Premier League seasons.

### Split Design

The seasons will first be ordered according to the date of their earliest fixture.

The dataset will then be divided as follows:

- **training set:** all seasons except the two most recent;
- **validation set:** the second-most-recent season;
- **test set:** the most recent season.

With ten seasons of data, this corresponds to:

- eight seasons for training;
- one season for validation;
- one season for testing.

Let the ordered seasons be:

$$
S_1,S_2,\ldots,S_T.
$$

The training data are:

$$
\mathcal{D}_{\text{train}}
=
\bigcup_{t=1}^{T-2}\mathcal{D}_{S_t},
$$

the validation data are:

$$
\mathcal{D}_{\text{validation}}
=
\mathcal{D}_{S_{T-1}},
$$

and the test data are:

$$
\mathcal{D}_{\text{test}}
=
\mathcal{D}_{S_T}.
$$

### Role of Each Dataset

The training set will be used to:

- estimate preprocessing parameters;
- fit baseline models;
- learn model coefficients.

The validation set will be used to:

- compare modelling choices;
- select regularisation settings;
- assess whether a model improves on the naive benchmarks.

The test set must remain untouched during model development. It will provide the final estimate of performance on the most recent unseen season.

### Preprocessing Discipline

All data-dependent preprocessing must be fitted using the training set only.

This includes:

- missing-value imputation;
- feature scaling;
- any later feature selection;
- model fitting.

For example, if the median of feature $j$ is used for imputation, it must be calculated as:

$$
\widetilde{x}_{j,\text{train}}
=
\operatorname{median}
\left(
X_{\text{train},j}
\right),
$$

and then applied unchanged to the validation and test sets.

Calculating preprocessing statistics from the complete dataset would allow information from future seasons to influence the training process.

### Split Validation

The split will be checked to confirm that:

- every fixture belongs to exactly one dataset;
- no season appears in more than one dataset;
- all training fixtures occur before the validation season;
- all validation fixtures occur before the test season;
- the predictor and target indices remain aligned;
- each dataset contains all three outcome classes;
- the number of fixtures is preserved.

This section will create:

- `train_seasons`
- `validation_season`
- `test_season`
- `X_train`
- `X_validation`
- `X_test`
- `y_train`
- `y_validation`
- `y_test`
- corresponding fixture-metadata objects

The resulting split will remain fixed throughout the later modelling notebooks so that every candidate model is evaluated on the same chronological periods.

In [ ]:
# ============================================================
# 4. Chronological Train–Validation–Test Split
# ============================================================

# ------------------------------------------------------------
# Order seasons by the date of their earliest fixture
# ------------------------------------------------------------

season_date_summary = (
    model_data
    .groupby(
        season_column,
        as_index=False,
    )
    .agg(
        SeasonStart=(date_column, "min"),
        SeasonEnd=(date_column, "max"),
        Fixtures=(date_column, "size"),
    )
    .sort_values(
        by="SeasonStart",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ordered_seasons = (
    season_date_summary[season_column]
    .tolist()
)

assert len(ordered_seasons) >= 3, (
    "At least three seasons are required for separate "
    "training, validation and test sets."
)


# ------------------------------------------------------------
# Define the fixed chronological split
# ------------------------------------------------------------

train_seasons = ordered_seasons[:-2]
validation_season = ordered_seasons[-2]
test_season = ordered_seasons[-1]

assert train_seasons, (
    "The training set must contain at least one season."
)

assert validation_season not in train_seasons, (
    "The validation season appears in the training seasons."
)

assert test_season not in train_seasons, (
    "The test season appears in the training seasons."
)

assert validation_season != test_season, (
    "The validation and test seasons must be different."
)


# ------------------------------------------------------------
# Create split masks
# ------------------------------------------------------------

train_mask = model_data[season_column].isin(
    train_seasons
)

validation_mask = (
    model_data[season_column] == validation_season
)

test_mask = (
    model_data[season_column] == test_season
)


# ------------------------------------------------------------
# Confirm that every fixture belongs to exactly one split
# ------------------------------------------------------------

split_membership_count = (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

assert (split_membership_count == 1).all(), (
    "Every fixture must belong to exactly one split."
)

assert int(train_mask.sum()) > 0, (
    "The training set is empty."
)

assert int(validation_mask.sum()) > 0, (
    "The validation set is empty."
)

assert int(test_mask.sum()) > 0, (
    "The test set is empty."
)

assert (
    int(train_mask.sum())
    + int(validation_mask.sum())
    + int(test_mask.sum())
    == len(model_data)
), "The split does not preserve every fixture."


# ------------------------------------------------------------
# Construct predictor, target and metadata splits
# ------------------------------------------------------------

X_train = X.loc[train_mask].copy()
X_validation = X.loc[validation_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_validation = y.loc[validation_mask].copy()
y_test = y.loc[test_mask].copy()

metadata_train = fixture_metadata.loc[
    train_mask
].copy()

metadata_validation = fixture_metadata.loc[
    validation_mask
].copy()

metadata_test = fixture_metadata.loc[
    test_mask
].copy()


# ------------------------------------------------------------
# Validate index alignment within each split
# ------------------------------------------------------------

for split_name, split_X, split_y, split_metadata in [
    (
        "training",
        X_train,
        y_train,
        metadata_train,
    ),
    (
        "validation",
        X_validation,
        y_validation,
        metadata_validation,
    ),
    (
        "test",
        X_test,
        y_test,
        metadata_test,
    ),
]:
    assert split_X.index.equals(split_y.index), (
        f"The {split_name} predictors and target are not aligned."
    )

    assert split_X.index.equals(split_metadata.index), (
        f"The {split_name} predictors and metadata are not aligned."
    )

    assert list(split_X.columns) == feature_columns, (
        f"The {split_name} predictor columns changed."
    )


# ------------------------------------------------------------
# Validate that seasons do not overlap across splits
# ------------------------------------------------------------

observed_train_seasons = set(
    metadata_train[season_column].unique()
)

observed_validation_seasons = set(
    metadata_validation[season_column].unique()
)

observed_test_seasons = set(
    metadata_test[season_column].unique()
)

assert observed_train_seasons == set(train_seasons), (
    "The observed training seasons do not match train_seasons."
)

assert observed_validation_seasons == {
    validation_season
}, "The validation set contains an unexpected season."

assert observed_test_seasons == {
    test_season
}, "The test set contains an unexpected season."

assert observed_train_seasons.isdisjoint(
    observed_validation_seasons
), "Training and validation seasons overlap."

assert observed_train_seasons.isdisjoint(
    observed_test_seasons
), "Training and test seasons overlap."

assert observed_validation_seasons.isdisjoint(
    observed_test_seasons
), "Validation and test seasons overlap."


# ------------------------------------------------------------
# Validate chronological separation
# ------------------------------------------------------------

training_end_date = metadata_train[
    date_column
].max()

validation_start_date = metadata_validation[
    date_column
].min()

validation_end_date = metadata_validation[
    date_column
].max()

test_start_date = metadata_test[
    date_column
].min()

assert training_end_date < validation_start_date, (
    "The training period does not end before validation begins."
)

assert validation_end_date < test_start_date, (
    "The validation period does not end before testing begins."
)

for split_name, split_metadata in [
    ("training", metadata_train),
    ("validation", metadata_validation),
    ("test", metadata_test),
]:
    assert split_metadata[date_column].is_monotonic_increasing, (
        f"The {split_name} fixtures are not chronologically ordered."
    )


# ------------------------------------------------------------
# Validate outcome classes in every split
# ------------------------------------------------------------

expected_classes = set(class_order)

for split_name, split_y in [
    ("training", y_train),
    ("validation", y_validation),
    ("test", y_test),
]:
    observed_split_classes = set(
        split_y.unique()
    )

    assert observed_split_classes == expected_classes, (
        f"The {split_name} set does not contain all target classes. "
        f"Observed: {sorted(observed_split_classes)}"
    )


# ------------------------------------------------------------
# Add a split label for later auditing
# ------------------------------------------------------------

split_labels = pd.Series(
    index=model_data.index,
    dtype="string",
    name="DatasetSplit",
)

split_labels.loc[train_mask] = "Train"
split_labels.loc[validation_mask] = "Validation"
split_labels.loc[test_mask] = "Test"

assert split_labels.notna().all(), (
    "At least one fixture has no dataset-split label."
)


# ------------------------------------------------------------
# Create split summary
# ------------------------------------------------------------

split_summary = pd.DataFrame(
    {
        "Split": [
            "Train",
            "Validation",
            "Test",
        ],
        "Seasons": [
            len(train_seasons),
            1,
            1,
        ],
        "SeasonRange": [
            (
                f"{train_seasons[0]} to "
                f"{train_seasons[-1]}"
            ),
            str(validation_season),
            str(test_season),
        ],
        "Fixtures": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            round(
                100 * len(X_train) / len(model_data),
                2,
            ),
            round(
                100 * len(X_validation) / len(model_data),
                2,
            ),
            round(
                100 * len(X_test) / len(model_data),
                2,
            ),
        ],
        "StartDate": [
            metadata_train[date_column].min().date(),
            metadata_validation[date_column].min().date(),
            metadata_test[date_column].min().date(),
        ],
        "EndDate": [
            metadata_train[date_column].max().date(),
            metadata_validation[date_column].max().date(),
            metadata_test[date_column].max().date(),
        ],
    }
)


# ------------------------------------------------------------
# Create class-distribution summary by split
# ------------------------------------------------------------

class_distribution_by_split = pd.concat(
    [
        pd.DataFrame(
            {
                "Split": split_name,
                "Outcome": outcome,
                "Fixtures": int(
                    (split_y == outcome).sum()
                ),
                "Percentage": round(
                    100
                    * (split_y == outcome).mean(),
                    2,
                ),
            },
            index=[0],
        )
        for split_name, split_y in [
            ("Train", y_train),
            ("Validation", y_validation),
            ("Test", y_test),
        ]
        for outcome in class_order
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Chronological data split completed successfully.")
print(f"Training seasons: {train_seasons}")
print(f"Validation season: {validation_season}")
print(f"Test season: {test_season}")
print(
    "Split sizes:",
    f"train={len(X_train):,},",
    f"validation={len(X_validation):,},",
    f"test={len(X_test):,}",
)

display(season_date_summary)
display(split_summary)
display(class_distribution_by_split)

### Results and Interpretation

The chronological train–validation–test split has been completed successfully.

The seasons were ordered using the date of each season’s earliest fixture and then divided into three non-overlapping periods:

- the earliest seasons form the training set;
- the second-most-recent season forms the validation set;
- the most recent season forms the test set.

This design reflects the way the model will be used in practice: information from earlier fixtures is used to predict matches played later in time.

The validation confirms that:

- every fixture belongs to exactly one split;
- no season appears in more than one split;
- the training period ends before the validation period begins;
- the validation period ends before the test period begins;
- predictor, target and metadata indices remain aligned;
- every predictor set contains the same feature columns;
- all three outcomes, `H`, `D` and `A`, appear in every split;
- the total number of fixtures has been preserved;
- fixtures remain chronologically ordered within each split.

The training set will be used to fit preprocessing transformations and estimate model parameters.

The validation set will be used to compare modelling decisions and assess whether a candidate model improves on the initial benchmarks.

The test set will remain untouched during model development. Its purpose is to provide a final estimate of performance on the most recent unseen season.

This separation is especially important for probability modelling. Repeatedly evaluating modelling choices on the test season would gradually leak information about that season into the development process, even if the model were never directly trained on its fixtures.

No imputation or scaling has yet been applied. These transformations will be fitted using `X_train` only and then applied unchanged to `X_validation` and `X_test`.

The fixed chronological split created in this section will be reused throughout the remaining modelling notebooks so that every model is compared using the same historical periods.

## 5. Target Distribution and Class Balance

Before constructing the first probability baselines, the distribution of match outcomes must be examined across the training, validation and test sets.

The target variable contains three possible outcomes:

$$
Y_i \in \{H,D,A\},
$$

where:

- `H` represents a home win;
- `D` represents a draw;
- `A` represents an away win.

Football match outcomes are not evenly distributed. Home wins are typically more common than draws or away wins because of the historical home advantage.

For dataset split $s$ and outcome class $k$, the observed class proportion is:

$$
\widehat{\pi}_{s,k}
=
\frac{N_{s,k}}{N_s},
$$

where:

- $N_{s,k}$ is the number of fixtures in split $s$ with outcome $k$;
- $N_s$ is the total number of fixtures in split $s$.

The class proportions will be calculated separately for:

- the training set;
- the validation set;
- the test set.

### Why Class Balance Matters

Class balance affects both modelling and evaluation.

A model that predicts the most common outcome too frequently may achieve a superficially reasonable accuracy while producing poor probability estimates for draws and away wins.

For example, predicting a home win for every fixture could outperform random classification on accuracy, but it would provide no meaningful assessment of uncertainty.

For this reason, accuracy will remain a secondary metric. The primary metrics will evaluate the complete predicted probability distribution.

### Distribution Shift

The outcome frequencies may change between seasons.

The validation and test sets may contain different proportions of home wins, draws and away wins from the training period because of:

- changes in home advantage;
- variation in team strength;
- unusual seasonal conditions;
- promoted and relegated teams;
- changes in playing style or league competitiveness.

These differences represent genuine temporal distribution shift and should not be corrected using information from future seasons.

### Historical-Frequency Baseline

The training-set outcome proportions will later define the historical-frequency baseline.

For outcome $k$:

$$
\widehat{p}_k
=
\frac{N_{\text{train},k}}
{N_{\text{train}}}.
$$

The same fixed training probabilities will then be assigned to every validation and test fixture.

It would be incorrect to calculate separate baseline probabilities using the validation or test outcomes because those results would not be known when the predictions were made.

### Section Objectives

This section will:

- count each outcome in every dataset split;
- calculate outcome percentages;
- compare class distributions across time;
- identify the majority class;
- quantify any change between training, validation and test periods;
- store the training-set class probabilities for later baseline forecasts.

The main objects created will be:

- `training_class_probabilities`
- `class_balance_summary`
- `class_balance_pivot`

These will provide the foundation for the naive probability benchmarks in the next section.

In [ ]:
# ============================================================
# 5. Target Distribution and Class Balance
# ============================================================

# ------------------------------------------------------------
# Store the target vectors for each chronological split
# ------------------------------------------------------------

target_splits = {
    "Train": y_train,
    "Validation": y_validation,
    "Test": y_test,
}


# ------------------------------------------------------------
# Calculate outcome counts and proportions by split
# ------------------------------------------------------------

class_balance_records = []

for split_name, split_target in target_splits.items():

    outcome_counts = (
        split_target
        .value_counts()
        .reindex(
            class_order,
            fill_value=0,
        )
    )

    outcome_proportions = (
        outcome_counts / len(split_target)
    )

    for outcome in class_order:
        class_balance_records.append(
            {
                "Split": split_name,
                "Outcome": outcome,
                "Fixtures": int(
                    outcome_counts.loc[outcome]
                ),
                "Proportion": float(
                    outcome_proportions.loc[outcome]
                ),
                "Percentage": round(
                    100
                    * outcome_proportions.loc[outcome],
                    2,
                ),
            }
        )


class_balance_summary = pd.DataFrame(
    class_balance_records
)


# ------------------------------------------------------------
# Validate the class-balance summary
# ------------------------------------------------------------

assert len(class_balance_summary) == (
    len(target_splits) * len(class_order)
), (
    "The class-balance summary does not contain every "
    "split-outcome combination."
)

for split_name, split_target in target_splits.items():

    split_summary = class_balance_summary[
        class_balance_summary["Split"] == split_name
    ]

    assert set(
        split_summary["Outcome"]
    ) == set(class_order), (
        f"The {split_name} summary does not contain all outcomes."
    )

    assert (
        split_summary["Fixtures"].sum()
        == len(split_target)
    ), (
        f"The {split_name} outcome counts do not match "
        "the number of fixtures."
    )

    assert np.isclose(
        split_summary["Proportion"].sum(),
        1.0,
    ), (
        f"The {split_name} class proportions do not sum to 1."
    )


# ------------------------------------------------------------
# Store training-set probabilities for the later baseline
# ------------------------------------------------------------

training_class_probabilities = (
    y_train
    .value_counts(normalize=True)
    .reindex(class_order)
    .astype(float)
)

assert training_class_probabilities.notna().all(), (
    "At least one target class is missing from the "
    "training probabilities."
)

assert np.isclose(
    training_class_probabilities.sum(),
    1.0,
), "Training class probabilities do not sum to 1."

assert (
    training_class_probabilities >= 0
).all(), "Training class probabilities cannot be negative."

assert (
    training_class_probabilities <= 1
).all(), "Training class probabilities cannot exceed 1."


# ------------------------------------------------------------
# Identify the majority outcome in each split
# ------------------------------------------------------------

majority_class_records = []

for split_name, split_target in target_splits.items():

    split_probabilities = (
        split_target
        .value_counts(normalize=True)
        .reindex(class_order)
    )

    majority_outcome = (
        split_probabilities.idxmax()
    )

    majority_class_records.append(
        {
            "Split": split_name,
            "MajorityOutcome": majority_outcome,
            "MajorityProportion": float(
                split_probabilities.loc[
                    majority_outcome
                ]
            ),
            "MajorityPercentage": round(
                100
                * split_probabilities.loc[
                    majority_outcome
                ],
                2,
            ),
        }
    )


majority_class_summary = pd.DataFrame(
    majority_class_records
)


# ------------------------------------------------------------
# Create a comparison table of class percentages
# ------------------------------------------------------------

class_balance_pivot = (
    class_balance_summary
    .pivot(
        index="Outcome",
        columns="Split",
        values="Percentage",
    )
    .reindex(
        index=class_order,
        columns=[
            "Train",
            "Validation",
            "Test",
        ],
    )
    .reset_index()
)


# ------------------------------------------------------------
# Quantify distribution shift relative to training
# ------------------------------------------------------------

class_balance_shift = (
    class_balance_pivot
    .copy()
)

class_balance_shift[
    "ValidationMinusTrain"
] = (
    class_balance_shift["Validation"]
    - class_balance_shift["Train"]
)

class_balance_shift[
    "TestMinusTrain"
] = (
    class_balance_shift["Test"]
    - class_balance_shift["Train"]
)

class_balance_shift[
    "ValidationMinusTrain"
] = class_balance_shift[
    "ValidationMinusTrain"
].round(2)

class_balance_shift[
    "TestMinusTrain"
] = class_balance_shift[
    "TestMinusTrain"
].round(2)


# ------------------------------------------------------------
# Create a compact training-probability table
# ------------------------------------------------------------

training_probability_table = pd.DataFrame(
    {
        "Outcome": class_order,
        "Probability": [
            training_class_probabilities.loc[outcome]
            for outcome in class_order
        ],
        "Percentage": [
            round(
                100
                * training_class_probabilities.loc[outcome],
                2,
            )
            for outcome in class_order
        ],
    }
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert (
    training_probability_table["Probability"] >= 0
).all(), "A baseline probability is negative."

assert (
    training_probability_table["Probability"] <= 1
).all(), "A baseline probability exceeds 1."

assert np.isclose(
    training_probability_table[
        "Probability"
    ].sum(),
    1.0,
), "The baseline probabilities do not sum to 1."


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Target distribution analysis completed successfully.")
print(
    "Training-set majority outcome:",
    majority_class_summary.loc[
        majority_class_summary["Split"] == "Train",
        "MajorityOutcome",
    ].iloc[0],
)
print(
    "Training class probabilities:",
    {
        outcome: round(
            training_class_probabilities.loc[outcome],
            4,
        )
        for outcome in class_order
    },
)

display(class_balance_summary)

display(class_balance_pivot)

display(class_balance_shift)

display(majority_class_summary)

display(training_probability_table)

### Results and Interpretation

The target-distribution analysis confirms that Premier League match outcomes are not evenly balanced across the three classes.

The training, validation and test sets each contain:

- home wins, represented by `H`;
- draws, represented by `D`;
- away wins, represented by `A`.

The `class_balance_summary` table reports the number and proportion of each outcome within every chronological split, while `class_balance_pivot` allows the percentages to be compared directly across time.

Home wins form the majority class in the training period. This reflects the historical home advantage present in Premier League football.

However, the class proportions are not identical across the training, validation and test seasons. The `class_balance_shift` table measures these changes in percentage points relative to the training set:

$$
\Delta_{s,k}
=
\widehat{\pi}_{s,k}
-
\widehat{\pi}_{\text{train},k},
$$

where $s$ represents either the validation or test set and $k$ represents one of the three outcomes.

These differences indicate genuine temporal distribution shift. The balance between home wins, draws and away wins varies from season to season because of changing teams, league competitiveness and broader football conditions.

The future models should not attempt to remove this shift using information from the validation or test outcomes. Instead, they must learn from the historical training period and be evaluated on how well they generalise to the later seasons.

The training-set class proportions have been stored in:

`training_class_probabilities`

These probabilities will form the historical-frequency benchmark in the next section.

For every future fixture, that benchmark will assign the same probability vector:

$$
\widehat{\mathbf{p}}_{\text{frequency}}
=
\left(
\widehat{p}_{H,\text{train}},
\widehat{p}_{D,\text{train}},
\widehat{p}_{A,\text{train}}
\right).
$$

Only the training outcomes are used to estimate these probabilities. The validation and test labels remain excluded from the forecasting process.

The majority-class percentages also demonstrate why accuracy alone is insufficient. A model that predicts the most common result for every fixture may achieve non-trivial accuracy, but it would fail to distinguish between matches and would provide poor probability estimates for less common outcomes.

The next section will construct and evaluate naive probability benchmarks using:

- uniform probabilities;
- training-set historical frequencies;
- majority-class predictions.

These benchmarks establish the minimum performance that the feature-based multinomial logistic-regression model must improve upon.

## 6. Naive Probability Benchmarks

Before fitting a feature-based model, several deliberately simple benchmarks must be evaluated.

These benchmarks provide reference levels for probability quality. A more complex model is only useful if it produces better out-of-sample forecasts than methods requiring little or no football information.

The benchmarks will be evaluated on both the validation and test sets using the same fixed chronological split established earlier.

### Uniform-Probability Baseline

The uniform baseline assigns equal probability to every possible outcome:

$$
\widehat{p}_H
=
\widehat{p}_D
=
\widehat{p}_A
=
\frac{1}{3}.
$$

Every fixture therefore receives the probability vector:

$$
\widehat{\mathbf{p}}_{\text{uniform}}
=
\left(
\frac{1}{3},
\frac{1}{3},
\frac{1}{3}
\right).
$$

This benchmark contains no information about football, home advantage, team strength or historical outcome frequencies.

Its purpose is to establish the performance of a completely uninformative probability forecast.

### Historical-Frequency Baseline

The historical-frequency baseline assigns probabilities using the outcome proportions observed in the training set.

For outcome class $k$:

$$
\widehat{p}_{k,\text{train}}
=
\frac{N_{k,\text{train}}}
{N_{\text{train}}},
$$

where:

- $N_{k,\text{train}}$ is the number of training fixtures with outcome $k$;
- $N_{\text{train}}$ is the total number of training fixtures.

Every validation and test fixture receives the same probability vector:

$$
\widehat{\mathbf{p}}_{\text{frequency}}
=
\left(
\widehat{p}_{H,\text{train}},
\widehat{p}_{D,\text{train}},
\widehat{p}_{A,\text{train}}
\right).
$$

This benchmark captures the unconditional historical home advantage and the long-run frequencies of draws and away wins.

It does not distinguish between individual fixtures.

Only training-set outcomes are used to estimate these probabilities. Recalculating them from the validation or test sets would use information that would not have been available when the predictions were made.

### Majority-Class Baseline

The majority-class benchmark predicts the most common training outcome as the most likely result for every fixture.

Let:

$$
k^*
=
\arg\max_{k \in \{H,D,A\}}
\widehat{p}_{k,\text{train}}.
$$

The predicted class is then:

$$
\widehat{Y}_i = k^*
$$

for every fixture $i$.

This benchmark is mainly useful for interpreting accuracy.

A majority-class classifier may achieve a non-trivial percentage of correct predictions despite making no distinction between fixtures. It is therefore not a meaningful probability model by itself.

For probability-based evaluation, the historical-frequency probabilities will be used rather than assigning probability one to the majority class and zero to the others. A deterministic probability vector would produce extremely large or undefined log loss whenever another outcome occurred.

### Evaluation Metrics

The uniform and historical-frequency probability forecasts will be evaluated using multiclass log loss:

$$
\operatorname{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
y_{i,k}
\log
\left(
\widehat{p}_{i,k}
\right).
$$

Lower values indicate better probability forecasts.

The multiclass Brier score will also be calculated:

$$
\operatorname{Brier}
=
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
\left(
\widehat{p}_{i,k}
-
y_{i,k}
\right)^2.
$$

Lower Brier scores indicate that the predicted probability vector lies closer to the observed one-hot outcome vector.

Accuracy will be reported as a secondary metric:

$$
\operatorname{Accuracy}
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbb{1}
\left(
\widehat{Y}_i = Y_i
\right).
$$

### Benchmark Requirements

The implementation will confirm that:

- each probability is between zero and one;
- every probability row sums to one;
- probability columns follow the fixed order `(H, D, A)`;
- historical probabilities are estimated from `y_train` only;
- validation and test predictions contain the correct number of fixtures;
- no validation or test outcomes are used to construct the forecasts;
- all metrics are calculated consistently across benchmarks and splits.

This section will create:

- uniform validation and test probabilities;
- historical-frequency validation and test probabilities;
- majority-class validation and test predictions;
- a reusable multiclass Brier-score function;
- a benchmark-results table.

These results will establish the minimum standard that multinomial logistic regression must exceed.

In [ ]:
# ============================================================
# 6. Naive Probability Benchmarks
# ============================================================

from sklearn.metrics import accuracy_score, log_loss


# ------------------------------------------------------------
# Reusable multiclass Brier-score function
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probability_matrix,
    labels,
):
    """
    Calculate the multiclass Brier score.

    The score is the mean squared distance between each predicted
    probability vector and the observed one-hot outcome vector.
    """
    y_true = pd.Series(y_true).reset_index(drop=True)

    probabilities = np.asarray(
        probability_matrix,
        dtype=float,
    )

    assert probabilities.shape == (
        len(y_true),
        len(labels),
    ), (
        "The probability matrix has an unexpected shape. "
        f"Expected {(len(y_true), len(labels))}, "
        f"received {probabilities.shape}."
    )

    label_to_position = {
        label: position
        for position, label in enumerate(labels)
    }

    invalid_labels = sorted(
        set(y_true.unique()) - set(labels)
    )

    assert not invalid_labels, (
        "The target contains labels that are not included in "
        f"the supplied class order: {invalid_labels}"
    )

    observed_one_hot = np.zeros_like(
        probabilities,
        dtype=float,
    )

    observed_positions = (
        y_true
        .map(label_to_position)
        .to_numpy()
    )

    observed_one_hot[
        np.arange(len(y_true)),
        observed_positions,
    ] = 1.0

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - observed_one_hot
                ) ** 2,
                axis=1,
            )
        )
    )


# ------------------------------------------------------------
# Reusable class-order-safe multiclass log-loss function
# ------------------------------------------------------------

def multiclass_log_loss(
    y_true,
    probability_frame,
    labels,
):
    """
    Calculate multiclass log loss while explicitly matching each
    observed outcome to its corresponding probability column.

    The project's display order is (H, D, A), whereas scikit-learn
    internally assumes lexicographic class ordering when interpreting
    probability-matrix columns. This helper avoids ambiguity by
    calculating the row-level loss directly, then independently checks
    the result against scikit-learn using explicitly sorted columns.
    """
    y_true = pd.Series(y_true).copy()

    assert isinstance(
        probability_frame,
        pd.DataFrame,
    ), "Log-loss probabilities must be stored in a pandas DataFrame."

    assert probability_frame.index.equals(
        y_true.index
    ), "The probability rows and target rows are not aligned."

    assert list(probability_frame.columns) == list(labels), (
        "The probability columns are not in the required project "
        f"order: {list(labels)}."
    )

    invalid_labels = sorted(
        set(y_true.dropna().unique()) - set(labels)
    )

    assert not invalid_labels, (
        "The target contains labels that are absent from the "
        f"probability columns: {invalid_labels}"
    )

    probability_values = probability_frame.to_numpy(dtype=float)

    assert np.isfinite(probability_values).all(), (
        "The probability frame contains non-finite values."
    )

    assert (probability_values >= 0).all(), (
        "The probability frame contains a value below zero."
    )

    assert (probability_values <= 1).all(), (
        "The probability frame contains a value above one."
    )

    assert np.allclose(
        probability_values.sum(axis=1),
        1.0,
    ), "The probability rows do not sum to one."

    column_positions = {
        label: position
        for position, label in enumerate(probability_frame.columns)
    }

    observed_positions = np.array(
        [column_positions[outcome] for outcome in y_true],
        dtype=int,
    )

    observed_probabilities = probability_values[
        np.arange(len(y_true)),
        observed_positions,
    ]

    observed_probabilities = np.clip(
        observed_probabilities,
        1e-15,
        1.0,
    )

    manual_log_loss = float(
        -np.mean(np.log(observed_probabilities))
    )

    sklearn_class_order = sorted(labels)

    sklearn_log_loss = log_loss(
        y_true,
        probability_frame[
            sklearn_class_order
        ].to_numpy(dtype=float),
        labels=sklearn_class_order,
    )

    assert np.isclose(
        manual_log_loss,
        sklearn_log_loss,
    ), (
        "The direct and scikit-learn log-loss calculations do not "
        "agree. Check the class order and row alignment."
    )

    return manual_log_loss


# ------------------------------------------------------------
# Helper function: validate probability forecasts
# ------------------------------------------------------------

def validate_probability_forecast(
    probability_frame,
    expected_index,
    labels,
    name,
):
    """
    Confirm that a probability forecast is correctly structured.
    """
    assert isinstance(
        probability_frame,
        pd.DataFrame,
    ), f"{name} must be stored as a pandas DataFrame."

    assert probability_frame.index.equals(
        expected_index
    ), f"{name} is not aligned with the target index."

    assert list(
        probability_frame.columns
    ) == list(labels), (
        f"{name} probability columns are not ordered as "
        f"{list(labels)}."
    )

    probability_values = probability_frame.to_numpy(
        dtype=float
    )

    assert np.isfinite(
        probability_values
    ).all(), f"{name} contains non-finite probabilities."

    assert (
        probability_values >= 0
    ).all(), f"{name} contains a probability below zero."

    assert (
        probability_values <= 1
    ).all(), f"{name} contains a probability above one."

    assert np.allclose(
        probability_values.sum(axis=1),
        1.0,
    ), f"The probability rows in {name} do not sum to one."


# ------------------------------------------------------------
# Helper function: repeat one probability vector for every match
# ------------------------------------------------------------

def create_constant_probability_forecast(
    probability_vector,
    target_index,
    labels,
):
    """
    Repeat a fixed probability vector across a target index.
    """
    ordered_vector = (
        pd.Series(probability_vector)
        .reindex(labels)
        .astype(float)
    )

    assert ordered_vector.notna().all(), (
        "The probability vector does not contain every class."
    )

    assert np.isclose(
        ordered_vector.sum(),
        1.0,
    ), "The supplied probability vector does not sum to one."

    repeated_values = np.tile(
        ordered_vector.to_numpy(),
        (len(target_index), 1),
    )

    return pd.DataFrame(
        repeated_values,
        index=target_index,
        columns=labels,
    )


# ------------------------------------------------------------
# Define the two fixed probability vectors
# ------------------------------------------------------------

uniform_probability_vector = pd.Series(
    {
        outcome: 1 / len(class_order)
        for outcome in class_order
    },
    dtype=float,
)

historical_probability_vector = (
    training_class_probabilities
    .reindex(class_order)
    .astype(float)
)


# Confirm that historical probabilities use training outcomes only.
recalculated_training_probabilities = (
    y_train
    .value_counts(normalize=True)
    .reindex(class_order)
    .astype(float)
)

assert np.allclose(
    historical_probability_vector.to_numpy(),
    recalculated_training_probabilities.to_numpy(),
), (
    "The historical-frequency probabilities do not match "
    "the training-set outcome frequencies."
)


# ------------------------------------------------------------
# Create validation probability forecasts
# ------------------------------------------------------------

uniform_validation_probabilities = (
    create_constant_probability_forecast(
        uniform_probability_vector,
        y_validation.index,
        class_order,
    )
)

historical_validation_probabilities = (
    create_constant_probability_forecast(
        historical_probability_vector,
        y_validation.index,
        class_order,
    )
)


# ------------------------------------------------------------
# Create test probability forecasts
# ------------------------------------------------------------

uniform_test_probabilities = (
    create_constant_probability_forecast(
        uniform_probability_vector,
        y_test.index,
        class_order,
    )
)

historical_test_probabilities = (
    create_constant_probability_forecast(
        historical_probability_vector,
        y_test.index,
        class_order,
    )
)


# ------------------------------------------------------------
# Validate every probability forecast
# ------------------------------------------------------------

for forecast_name, forecast_frame, target_vector in [
    (
        "uniform validation forecast",
        uniform_validation_probabilities,
        y_validation,
    ),
    (
        "historical validation forecast",
        historical_validation_probabilities,
        y_validation,
    ),
    (
        "uniform test forecast",
        uniform_test_probabilities,
        y_test,
    ),
    (
        "historical test forecast",
        historical_test_probabilities,
        y_test,
    ),
]:
    validate_probability_forecast(
        probability_frame=forecast_frame,
        expected_index=target_vector.index,
        labels=class_order,
        name=forecast_name,
    )


# ------------------------------------------------------------
# Create class predictions
# ------------------------------------------------------------

# pandas idxmax resolves equal uniform probabilities using the first
# class in the fixed order, so the uniform baseline predicts H.
uniform_validation_predictions = (
    uniform_validation_probabilities
    .idxmax(axis=1)
)

uniform_test_predictions = (
    uniform_test_probabilities
    .idxmax(axis=1)
)

historical_validation_predictions = (
    historical_validation_probabilities
    .idxmax(axis=1)
)

historical_test_predictions = (
    historical_test_probabilities
    .idxmax(axis=1)
)

majority_class = (
    historical_probability_vector
    .idxmax()
)

majority_validation_predictions = pd.Series(
    majority_class,
    index=y_validation.index,
    name="MajorityPrediction",
    dtype="string",
)

majority_test_predictions = pd.Series(
    majority_class,
    index=y_test.index,
    name="MajorityPrediction",
    dtype="string",
)


# ------------------------------------------------------------
# Reusable probability-benchmark evaluation function
# ------------------------------------------------------------

def evaluate_probability_benchmark(
    benchmark_name,
    split_name,
    y_true,
    probabilities,
):
    """
    Evaluate a multiclass probability benchmark.
    """
    validate_probability_forecast(
        probability_frame=probabilities,
        expected_index=y_true.index,
        labels=class_order,
        name=f"{benchmark_name} {split_name}",
    )

    predicted_classes = probabilities.idxmax(
        axis=1
    )

    return {
        "Benchmark": benchmark_name,
        "Split": split_name,
        "Fixtures": len(y_true),
        "LogLoss": multiclass_log_loss(
            y_true,
            probabilities,
            labels=class_order,
        ),
        "BrierScore": multiclass_brier_score(
            y_true,
            probabilities,
            labels=class_order,
        ),
        "Accuracy": accuracy_score(
            y_true,
            predicted_classes,
        ),
        "PredictedClass": (
            predicted_classes.iloc[0]
            if predicted_classes.nunique() == 1
            else "Varies"
        ),
    }


# ------------------------------------------------------------
# Evaluate uniform and historical-frequency forecasts
# ------------------------------------------------------------

benchmark_records = []

for benchmark_name, validation_probabilities, test_probabilities in [
    (
        "Uniform Probability",
        uniform_validation_probabilities,
        uniform_test_probabilities,
    ),
    (
        "Historical Frequency",
        historical_validation_probabilities,
        historical_test_probabilities,
    ),
]:
    benchmark_records.append(
        evaluate_probability_benchmark(
            benchmark_name=benchmark_name,
            split_name="Validation",
            y_true=y_validation,
            probabilities=validation_probabilities,
        )
    )

    benchmark_records.append(
        evaluate_probability_benchmark(
            benchmark_name=benchmark_name,
            split_name="Test",
            y_true=y_test,
            probabilities=test_probabilities,
        )
    )


benchmark_results = pd.DataFrame(
    benchmark_records
)

for metric_column in [
    "LogLoss",
    "BrierScore",
    "Accuracy",
]:
    benchmark_results[metric_column] = (
        benchmark_results[metric_column]
        .astype(float)
        .round(6)
    )


# ------------------------------------------------------------
# Evaluate the majority-class accuracy benchmark
# ------------------------------------------------------------

majority_class_results = pd.DataFrame(
    [
        {
            "Benchmark": "Majority Class",
            "Split": "Validation",
            "Fixtures": len(y_validation),
            "PredictedClass": majority_class,
            "Accuracy": accuracy_score(
                y_validation,
                majority_validation_predictions,
            ),
        },
        {
            "Benchmark": "Majority Class",
            "Split": "Test",
            "Fixtures": len(y_test),
            "PredictedClass": majority_class,
            "Accuracy": accuracy_score(
                y_test,
                majority_test_predictions,
            ),
        },
    ]
)

majority_class_results["Accuracy"] = (
    majority_class_results["Accuracy"]
    .round(6)
)


# ------------------------------------------------------------
# Create a table of the benchmark probability vectors
# ------------------------------------------------------------

benchmark_probability_vectors = pd.DataFrame(
    {
        "Benchmark": [
            "Uniform Probability",
            "Historical Frequency",
        ],
        "H": [
            uniform_probability_vector.loc["H"],
            historical_probability_vector.loc["H"],
        ],
        "D": [
            uniform_probability_vector.loc["D"],
            historical_probability_vector.loc["D"],
        ],
        "A": [
            uniform_probability_vector.loc["A"],
            historical_probability_vector.loc["A"],
        ],
    }
)

for outcome in class_order:
    benchmark_probability_vectors[outcome] = (
        benchmark_probability_vectors[outcome]
        .round(6)
    )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert set(
    benchmark_results["Benchmark"]
) == {
    "Uniform Probability",
    "Historical Frequency",
}, "An expected probability benchmark is missing."

assert set(
    benchmark_results["Split"]
) == {
    "Validation",
    "Test",
}, "An expected evaluation split is missing."

assert (
    benchmark_results["LogLoss"] >= 0
).all(), "Log loss cannot be negative."

assert (
    benchmark_results["BrierScore"] >= 0
).all(), "Brier score cannot be negative."

assert benchmark_results[
    "Accuracy"
].between(0, 1).all(), (
    "Benchmark accuracy must remain between zero and one."
)

assert majority_class in class_order, (
    "The majority class is not a valid outcome."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Naive probability benchmarks evaluated successfully.")
print(f"Training-set majority class: {majority_class}")
print(
    "Historical probability vector:",
    {
        outcome: round(
            historical_probability_vector.loc[outcome],
            4,
        )
        for outcome in class_order
    },
)

display(benchmark_probability_vectors)

display(
    benchmark_results.sort_values(
        by=[
            "Split",
            "LogLoss",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

display(majority_class_results)

### Results and Interpretation

The naive probability benchmarks establish the minimum performance that every nonlinear model must exceed.

The uniform benchmark assigns:

$$
\widehat{\mathbf{p}}_{\mathrm{uniform}}
=
\left(
\frac{1}{3},
\frac{1}{3},
\frac{1}{3}
\right),
$$

while the historical-frequency benchmark assigns the outcome proportions estimated from the training period only.

The code above evaluates both benchmarks using the same class-order-safe log-loss function that was verified in Notebook 3. The displayed project probability order remains `(H, D, A)`, and each observed result is matched directly to its corresponding probability column.

The benchmark table should be interpreted as follows:

- lower log loss indicates better probabilistic forecasts;
- lower Brier score indicates smaller average squared probability error;
- accuracy is secondary because it ignores the quality of the full probability distribution;
- validation results may be used during model development;
- test results must not determine model families or hyperparameters.

The next stage prepares the predictor matrices for tree-based models. Unlike logistic regression, these models do not require standard scaling.

## 7. Preprocessing for Nonlinear Tree Models

The nonlinear models require complete numeric predictor matrices, but they do not require the standard scaling used for logistic regression.

### Median imputation

Several engineered predictors contain intentional missing values, particularly during the opening fixtures of a season before sufficient historical information exists.

For predictor $j$, the training-set median is:

$$
\widetilde{x}_{j,\mathrm{train}}
=
\operatorname{median}
\left(
X_{\mathrm{train},j}
\right).
$$

Every missing value in the training, validation and test matrices will be replaced using that same training-derived median:

$$
x_{i,j}^{\mathrm{imputed}}
=
\widetilde{x}_{j,\mathrm{train}}.
$$

No validation or test information is used when estimating the imputation values.

### Why standard scaling is omitted

Random Forest and gradient-boosted tree models make decisions by comparing a feature with candidate split thresholds. Rescaling a feature changes its numerical units but not the ordering of its observations, so standardisation is generally unnecessary.

The preprocessing sequence is therefore:

$$
\text{Raw predictors}
\longrightarrow
\text{Training-fitted median imputation}.
$$

The main objects created will be:

- `nonlinear_imputer`;
- `X_train_imputed`;
- `X_validation_imputed`;
- `X_test_imputed`;
- `imputed_feature_names`.

These matrices preserve the original feature values apart from the leakage-safe replacement of missing observations.

In [ ]:
# ============================================================
# 7. Preprocessing for Nonlinear Tree Models
# ============================================================

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# Record the original data state
# ------------------------------------------------------------

original_split_shapes = {
    "Train": X_train.shape,
    "Validation": X_validation.shape,
    "Test": X_test.shape,
}

original_missing_counts = {
    "Train": int(X_train.isna().sum().sum()),
    "Validation": int(X_validation.isna().sum().sum()),
    "Test": int(X_test.isna().sum().sum()),
}


# ------------------------------------------------------------
# Confirm every predictor contains training information
# ------------------------------------------------------------

all_missing_training_features = [
    column
    for column in feature_columns
    if X_train[column].isna().all()
]

assert not all_missing_training_features, (
    "Median imputation cannot be fitted because these features "
    "are completely missing in the training set: "
    f"{all_missing_training_features}"
)


# ------------------------------------------------------------
# Fit the imputer using the training period only
# ------------------------------------------------------------

nonlinear_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)

X_train_imputed_array = nonlinear_imputer.fit_transform(
    X_train
)

X_validation_imputed_array = nonlinear_imputer.transform(
    X_validation
)

X_test_imputed_array = nonlinear_imputer.transform(
    X_test
)


# ------------------------------------------------------------
# Restore labelled DataFrame structure
# ------------------------------------------------------------

imputed_feature_names = feature_columns.copy()

X_train_imputed = pd.DataFrame(
    X_train_imputed_array,
    index=X_train.index,
    columns=imputed_feature_names,
)

X_validation_imputed = pd.DataFrame(
    X_validation_imputed_array,
    index=X_validation.index,
    columns=imputed_feature_names,
)

X_test_imputed = pd.DataFrame(
    X_test_imputed_array,
    index=X_test.index,
    columns=imputed_feature_names,
)


# ------------------------------------------------------------
# Validate dimensions, columns and index alignment
# ------------------------------------------------------------

for split_name, original_X, imputed_X, target_y in [
    ("Train", X_train, X_train_imputed, y_train),
    (
        "Validation",
        X_validation,
        X_validation_imputed,
        y_validation,
    ),
    ("Test", X_test, X_test_imputed, y_test),
]:
    assert imputed_X.shape == original_X.shape, (
        f"The {split_name.lower()} matrix shape changed "
        "during imputation."
    )

    assert list(imputed_X.columns) == feature_columns, (
        f"The {split_name.lower()} feature order changed."
    )

    assert imputed_X.index.equals(original_X.index), (
        f"The {split_name.lower()} fixture index changed."
    )

    assert imputed_X.index.equals(target_y.index), (
        f"The {split_name.lower()} predictors and target "
        "are not aligned."
    )

    assert not imputed_X.isna().any().any(), (
        f"Missing values remain in the {split_name.lower()} "
        "matrix after imputation."
    )

    assert np.isfinite(imputed_X.to_numpy()).all(), (
        f"The {split_name.lower()} matrix contains "
        "non-finite values after imputation."
    )


# ------------------------------------------------------------
# Confirm the original matrices were not modified
# ------------------------------------------------------------

for split_name, original_X in [
    ("Train", X_train),
    ("Validation", X_validation),
    ("Test", X_test),
]:
    assert original_X.shape == original_split_shapes[split_name], (
        f"The original {split_name.lower()} shape changed."
    )

    assert int(
        original_X.isna().sum().sum()
    ) == original_missing_counts[split_name], (
        f"The original {split_name.lower()} missing-value "
        "pattern changed."
    )


# ------------------------------------------------------------
# Create preprocessing audit tables
# ------------------------------------------------------------

imputation_audit = pd.DataFrame(
    {
        "Feature": imputed_feature_names,
        "TrainingMissingValues": [
            int(X_train[column].isna().sum())
            for column in imputed_feature_names
        ],
        "TrainingMedian": nonlinear_imputer.statistics_,
    }
)

preprocessing_summary = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Fixtures": [
            len(X_train_imputed),
            len(X_validation_imputed),
            len(X_test_imputed),
        ],
        "Features": [
            X_train_imputed.shape[1],
            X_validation_imputed.shape[1],
            X_test_imputed.shape[1],
        ],
        "MissingBefore": [
            original_missing_counts["Train"],
            original_missing_counts["Validation"],
            original_missing_counts["Test"],
        ],
        "MissingAfter": [
            int(X_train_imputed.isna().sum().sum()),
            int(X_validation_imputed.isna().sum().sum()),
            int(X_test_imputed.isna().sum().sum()),
        ],
    }
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Nonlinear-model preprocessing completed successfully.")
print(f"Predictors retained: {len(imputed_feature_names)}")
print("Standard scaling applied: No")

display(preprocessing_summary)
display(imputation_audit.head(25))
display(X_train_imputed.head(10))

### Results and Interpretation

The nonlinear-model predictor matrices have been prepared successfully.

Median imputation was fitted using `X_train` only and then applied unchanged to the validation and test periods. This preserves the chronological evaluation structure and prevents future information from influencing the transformation.

The validation checks confirm that:

- all predictor columns were retained in their original order;
- fixture indices remain aligned with the target vectors;
- no missing or infinite values remain;
- the original unprocessed matrices were not altered;
- no standard scaling was applied.

The resulting matrices are:

- `X_train_imputed`;
- `X_validation_imputed`;
- `X_test_imputed`.

These are the inputs that should be used for the Random Forest and gradient-boosted tree models.

## 8. Validation Benchmark and Nonlinear-Modelling Setup

Notebook 3's tuned multinomial logistic-regression model is the principal feature-based benchmark for this notebook.

Only the validation result should influence nonlinear model selection:

$$
\operatorname{LogLoss}_{\mathrm{logistic,validation}}
=
0.934551.
$$

A nonlinear model should not be preferred merely because it has higher accuracy. It must improve, or at least credibly match, the probability metrics while remaining stable and interpretable.

In [ ]:
# ============================================================
# 8. Validation Benchmark and Nonlinear-Modelling Setup
# ============================================================

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterGrid


# ------------------------------------------------------------
# Fixed modelling configuration
# ------------------------------------------------------------

random_state = 42

logistic_validation_reference = pd.DataFrame(
    {
        "Model": ["Tuned multinomial logistic regression"],
        "ValidationLogLoss": [0.934551],
        "ValidationBrierScore": [0.549322],
        "ValidationAccuracy": [0.592105],
    }
)

# Each fitted nonlinear specification will later append one record here.
nonlinear_validation_records = []

print("Notebook 4 setup is complete.")
print("Next model: Random Forest probability baseline.")

display(logistic_validation_reference)

### Setup Complete

The notebook now reproduces the full leakage-safe foundation from Notebook 3:

- processed dataset loading and validation;
- predictor and target definition;
- fixed chronological train–validation–test split;
- class-distribution analysis;
- corrected probability-evaluation helpers;
- naive probability benchmarks;
- tree-appropriate median imputation;
- the tuned logistic-regression validation reference.

The next section will fit an untuned Random Forest baseline before any hyperparameter search is introduced.

## 9. Random Forest Probability Baseline